# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
  
`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

This dataset includes survey data on mental health indicators among residents of Kilifi County, Kenya, along with demographic details and psychological assessment scores such as GAD-7, PHQ-9, and PSQ.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL and load it
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'
dataset = mlc.Dataset(croissant_url)

# Metadata is an object; use its attributes directly
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Keywords: {dataset.metadata.keywords}")
print(f"Published: {dataset.metadata.datePublished}")
# Show available record sets
print(f"\nRecord sets found in dataset: {len(dataset.metadata.recordSets)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**All entities are referenced by their `@id` fields for consistency.**

In [ ]:
# List all record sets and their fields by @id
pprint.pprint([{'name': rs.name, '@id': rs.id, 'fields': [f.id for f in rs.fields]} for rs in dataset.metadata.recordSets])

### Example: Print a few records from the main survey record set
Replace `<main_record_set_id>` below with the actual `@id` for your primary survey record set.

In [ ]:
# List all record set ids (reference by `@id`)
all_record_set_ids = [rs.id for rs in dataset.metadata.recordSets]
print("All record sets by @id:", all_record_set_ids)

# Let's assume the main record set is the first one
main_record_set_id = all_record_set_ids[0]

print(f"Sample records from record set {main_record_set_id}:")
for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
    pprint.pprint(record)
    if i >= 2:
        break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set into a pandas DataFrame (by @id)
dataframes = {}
for record_set in dataset.metadata.recordSets:
    print(f"Loading record set '{record_set.name}' (@id={record_set.id}) ...")
    records = list(dataset.records(record_set=record_set.id))
    if records:
        dataframes[record_set.id] = pd.DataFrame(records)
        print(f"  Columns: {dataframes[record_set.id].columns.tolist()}")
        print(f"  Number of records: {len(records)}\n")
    else:
        print(f"  (empty)\n")

# Preview the primary survey DataFrame
main_df = dataframes.get(main_record_set_id)
if main_df is not None:
    print(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All fields should be referenced by their `@id` as shown in overview.

In [ ]:
# Suppose one of the numeric variables is 'gad7_total_score', reference by its @id (e.g., 'gad7_total_score')
# Replace <numeric_field_id> below with exact @id (column name) for the numeric field you wish to analyze
numeric_field = [col for col in main_df.columns if 'gad7' in col or 'phq9' in col or 'psq' in col]

if numeric_field:
    numeric_field = numeric_field[0]
else:
    numeric_field = main_df.columns[0]  # fallback, pick first column

print(f"Chosen numeric field for analysis: {numeric_field}")

# Filtering (e.g., total score > 10)
threshold = 10
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalizing selected numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Pick a grouping field (e.g., 'gender', by @id)
group_fields = [col for col in main_df.columns if 'gender' in col or 'age' in col or 'village' in col]
group_field = group_fields[0] if group_fields else None

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by {group_field} (showing average {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the chosen numeric field
plt.figure(figsize=(8, 5))
sns.histplot(main_df[numeric_field].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

if group_field:
    # Violin plot or boxplot by group
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=main_df[group_field], y=main_df[numeric_field])
    plt.xticks(rotation=30)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² mental health survey dataset from Kilifi County, Kenya, using the `mlcroissant` library.

- We loaded the dataset via its Croissant schema and reviewed its record sets referenced by their `@id`s.
- We extracted the main survey records and examined demographic and mental health assessment fields.
- Exploratory data analysis included filtering, normalization, grouping, and visualizations.

**Key findings:**
- The dataset covers a rich set of demographic and psychological health variables suitable for statistical and machine learning analyses.
- Simple filtering and grouping highlight the value of structured metadata and standardized column references for reproducible research.

Next steps might include deeper statistical analysis, predictive modeling, or integration with other FAIR datasets.